In [1]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
import torch
from pathlib import Path
import pandas as pd
import ast
from PIL import Image, ImageDraw, ImageFont
import math
from typing import List

In [2]:
from peft import PeftModel, PeftConfig

In [3]:
from huggingface_hub import notebook_login

notebook_login()

In [3]:
COLOR_MAPPING = {
    (0xFF, 0xD7, 0x00): 'common room',
    (0xFF, 0xA5, 0x00): 'master room',
    (0xEE, 0xE8, 0xAA): 'living room',
    (0x6B, 0x8E, 0x23): 'balcony',
    (0xAD, 0xD8, 0xE6): 'bathroom',
    (0xF0, 0x80, 0x80): 'kitchen',
    (0xDD, 0xA0, 0xDD): 'storage',
    (0xDA, 0x70, 0xD6): 'dining',
}

ANNOT_DIR = Path("../../annotation/human_annotated_tags")

In [4]:
# class LocalVisionLLM:
#     def __init__(self, model_id: str, device: str = 'cuda'):
#         # Processor for both vision & text
#         self.processor = AutoProcessor.from_pretrained(
#             model_id,
#             use_auth_token=True,
#             trust_remote_code=True
#         )

#         # Vision‑language conditional generation model
#         self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
#             model_id,
#             torch_dtype=torch.float16,
#             device_map='auto',
#             trust_remote_code=True,
#             use_auth_token=True
#         )
#         # track actual device (could be spread across GPUs)
#         self.device = next(self.model.parameters()).device

#     def __call__(self, images: list[Image.Image], prompt: str, **generate_kwargs) -> str:
#         # 1) Build the “chat” messages list
#         messages = [{"type": "text",  "content": prompt}]
#         messages += [{"type": "image", "content": img} for img in images]

#         # 2) Turn messages into a single text string with the model’s chat template
#         #    (this adds any necessary <|user|>, <|assistant|> tokens and the generation prompt)
#         chat_text = self.processor.apply_chat_template(
#             messages,
#             tokenize=False,
#             add_generation_prompt=True
#         )

#         # 3) Extract visual inputs (pixel buffers, bboxes, etc.)
#         image_inputs, video_inputs = process_vision_info(messages)  # returns two things; video_inputs will be None here :contentReference[oaicite:0]{index=0}

#         # 4) Tokenize the chat text
#         text_inputs = self.processor(
#             chat_text,
#             return_tensors="pt",
#             add_special_tokens=False  # tokens already handled by apply_chat_template
#         )

#         # 5) Merge text + images into one input dict
#         #    (drop video_inputs since you’re only doing stills)
#         inputs = {**text_inputs, **(image_inputs or {})}
#         inputs = {k: v.to(self.device) for k, v in inputs.items()}

#         # 6) Generate
#         defaults = dict(max_new_tokens=256, do_sample=False)
#         outputs = self.model.generate(**inputs, **{**defaults, **generate_kwargs})

#         # 7) Decode & strip off the echoed prompt
#         full = self.processor.decode(outputs[0], skip_special_tokens=True)
#         return full[len(prompt):].strip()

In [4]:

class LocalVisionLLM:
    def __init__(self, model_id: str, device_map: str = 'auto', torch_dtype="auto"):
        # 1) Load processor
        self.processor = AutoProcessor.from_pretrained(
            model_id,
            use_auth_token=True,
            trust_remote_code=True
        )
        # 2) Load model
        self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            model_id,
            torch_dtype=torch_dtype,
            device_map=device_map,
            trust_remote_code=True,
            use_auth_token=True
        )
        # track actual device
        self.device = next(self.model.parameters()).device

    def __call__(self, images: list[Image.Image], prompt: str, max_new_tokens: int = 256, **gen_kwargs) -> str:
        # Build a single “user” message that contains both the images and the text
        messages = [
            {
                "role": "user",
                "content": [
                    *[
                        {"type": "image", "image": img}
                        for img in images
                    ],
                    {"type": "text", "text": prompt}
                ],
            }
        ]

        # 1) Format chat
        chat_text = self.processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        # 2) Extract vision inputs
        image_inputs, video_inputs = process_vision_info(messages)

        # 3) Prepare model inputs (batched)
        model_inputs = self.processor(
            text=[chat_text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        )
        model_inputs = {k: v.to(self.device) for k, v in model_inputs.items()}

        # 4) Generate
        gen_kwargs = dict(max_new_tokens=max_new_tokens, **gen_kwargs)
        generated_ids = self.model.generate(**model_inputs, **gen_kwargs)

        # 5) Trim off the prompt tokens
        #    generated_ids is shape [batch, seq_len]; inputs.input_ids is [batch, seq_len_in]
        input_ids = model_inputs["input_ids"]
        trimmed = [
            out_ids[input_ids.shape[1]:]
            for out_ids in generated_ids
        ]

        # 6) Decode
        output_texts = self.processor.batch_decode(
            trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )
        # single-example batch, so take [0]
        return output_texts[0].strip()

In [3]:
class PeftVisionLLM:
    def __init__(
        self,
        model_id: str,
        adapter_dir: str,                # <- path to your LoRA (folder containing adapter_model.safetensors)
        device_map: str = "auto",
        torch_dtype: str | torch.dtype = "auto",
        merge_adapters: bool = False,    # <- set True to fuse LoRA into base for faster inference
        use_auth_token: bool | str = True,
    ):
        # 1) Load processor (unchanged)
        self.processor = AutoProcessor.from_pretrained(
            model_id,
            use_auth_token=use_auth_token,
            trust_remote_code=True
        )

        # 2) Load base model
        base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            model_id,
            torch_dtype=torch_dtype,
            device_map=device_map,
            trust_remote_code=True,
            use_auth_token=use_auth_token
        )

        # 3) Attach LoRA adapter
        #    (Checks that adapter matches the base; raises helpful error if not.)
        self.model = PeftModel.from_pretrained(base, adapter_dir)

        # 4) Optionally merge for lower latency (no need if you plan to hot-swap adapters)
        if merge_adapters:
            # After merging, LoRA layers are fused into the base model and detached.
            self.model = self.model.merge_and_unload()

        self.model.eval()
        self.device = next(self.model.parameters()).device

    def __call__(
        self,
        images: List[Image.Image],
        prompt: str,
        max_new_tokens: int = 256,
        **gen_kwargs
    ) -> str:
        messages = [
            {
                "role": "user",
                "content": [
                    *[{"type": "image", "image": img} for img in images],
                    {"type": "text", "text": prompt},
                ],
            }
        ]

        # 1) Format chat
        chat_text = self.processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        # 2) Extract vision inputs
        image_inputs, video_inputs = process_vision_info(messages)

        # 3) Prepare inputs
        model_inputs = self.processor(
            text=[chat_text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        )
        model_inputs = {k: v.to(self.device) for k, v in model_inputs.items()}

        # 4) Generate
        gen_kwargs = dict(max_new_tokens=max_new_tokens, **gen_kwargs)
        with torch.inference_mode():
            generated_ids = self.model.generate(**model_inputs, **gen_kwargs)

        # 5) Trim prompt tokens
        input_ids = model_inputs["input_ids"]
        trimmed = [out_ids[input_ids.shape[1]:] for out_ids in generated_ids]

        # 6) Decode
        output_texts = self.processor.batch_decode(
            trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )
        return output_texts[0].strip()

In [5]:
model_id = "Qwen/Qwen2.5-VL-7B-Instruct"
vlm = LocalVisionLLM(model_id)

/home/airlay88/planscape/venv/lib/python3.10/site-packages/transformers/models/auto/processing_auto.py:262: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.
/home/airlay88/planscape/venv/lib

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [4]:
model_id = "Qwen/Qwen2.5-VL-7B-Instruct"
adapter_dir = "qwen2p5_vl_odd1out" 
vlm = PeftVisionLLM(model_id, adapter_dir)

/home/airlay88/planscape/venv/lib/python3.10/site-packages/transformers/models/auto/processing_auto.py:262: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.
/home/airlay88/planscape/venv/lib

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [8]:
task = "same_second"
legend = False

if legend:
    sfx = "_wlegend"
else:
    sfx = ""
# task = "same_second"
# task = "same_first_global"
# csv_path = Path("groups_diff_first_results.csv")
# csv_path = Path("groups_same_second_results.csv")
csv_path = Path("groups_{}_results.csv".format(task))
data_dir = Path("./randomized_options/examples_{}{}".format(task, sfx))
df = pd.read_csv(csv_path)

In [7]:

# def build_prompt(options: list[str]) -> str:
#     text = (
#         "I am showing you 5 apartment floorplan images.\n"
#         "4 share the same layout and 1 is different.\n\n"
#         "Please look carefully at spatial relationships, room types, and sizes.\n"
#         "Which example is the different one, and why?\n\n"
#         "Structure your reply exactly like:\n"
#         "1. **Different example:** <ID>\n"
#         "2. **Why:** <brief reasoning>\n\n"
#         "Examples by number:\n"
#     )
#     for idx, pid in enumerate(options, start=1):
#         text += f"- Example {idx}: ID {pid}\n"
#     return text

def build_prompt(options: list[str], legend: bool) -> str:
    COLOR_MAPPING = {
        (0xFF, 0xD7, 0x00): 'common room',
        (0xFF, 0xA5, 0x00): 'master room',
        (0xEE, 0xE8, 0xAA): 'living room',
        (0x6B, 0x8E, 0x23): 'balcony',
        (0xAD, 0xD8, 0xE6): 'bathroom',
        (0xF0, 0x80, 0x80): 'kitchen',
        (0xDD, 0xA0, 0xDD): 'storage',
        (0xDA, 0x70, 0xD6): 'dining',
    }

    text = (
        "I am showing you five apartment floorplans, labeled A through E.\n"
        "One of these plans has a different underlying floorplan pattern, while the other four share the same pattern.\n\n"
        "The thick black outline of each of the floorplans indicates the boundary of that floorplan. "
        "The red bar drawn on the black outline of each of the floorplans marks the main entrance of that floorplan.\n"
    )

    if legend:
        text += "Below is a color mapping indicating the room type for each fill color:\n"
        for rgb, room in COLOR_MAPPING.items():
            hex_code = f"#{rgb[0]:02X}{rgb[1]:02X}{rgb[2]:02X}"
            text += f"- {room}: {hex_code} (RGB{rgb})\n"
        text += "\n"

    text += (
        "Examine each floorplan **only within its thick black outer boundary**, and **use the main entrance (the red bar) "
        "as the point of entry to reorient yourself when reasoning about the spatial layout**, "
        "focusing on spatial layout, room types, and relative sizes.\n"
        "Question: Which floorplan (A, B, C, D, or E) has a different underlying pattern, and why?\n\n"
        "Please structure your response exactly as follows:\n"
        "1. **Different floorplan:** <A/B/C/D/E>\n"
        "2. **Why:** <brief reasoning>\n\n"
    )

    return text


def load_annotation(pid: str) -> str:
    """
    Load the human-annotated textual description for a given floorplan ID.
    """
    txt_path = ANNOT_DIR / f"{pid}.txt"
    try:
        return txt_path.read_text(encoding='utf-8').strip()
    except FileNotFoundError:
        return "<No annotation available>"

def build_2modal_prompt(options: list[str], legend) -> str:
    letters = ['A', 'B', 'C', 'D', 'E']
    COLOR_MAPPING = {
        (0xFF, 0xD7, 0x00): 'common room',
        (0xFF, 0xA5, 0x00): 'master room',
        (0xEE, 0xE8, 0xAA): 'living room',
        (0x6B, 0x8E, 0x23): 'balcony',
        (0xAD, 0xD8, 0xE6): 'bathroom',
        (0xF0, 0x80, 0x80): 'kitchen',
        (0xDD, 0xA0, 0xDD): 'storage',
        (0xDA, 0x70, 0xD6): 'dining',
    }

    text = (
        "I am presenting five apartment floorplans, labeled A through E.\n"
        "One plan has a different underlying floorplan pattern; the other four share the same design.\n\n"
        # "You can use the red bar (main entrance) to help reason about space and orientation.\n\n"
    )

    text  += (
        "Below are annotated descriptions corresponding to images labeled A–E. "
        "Please read them carefully.\n\n"
        "Descriptions:\n"
    )

    for label, pid in zip(letters, options):
        annotation = load_annotation(pid)
        text += f"  {label}. {annotation}\n"

    text += (
        "\nHere is the combined image showing floorplans A through E side by side.\n"
        "The thick black outline of each of the floorplans indicates the boundary of that floorplan. "
        "The red bar drawn on the black outline of each of the floorplans marks the main entrance of that floorplan. \n"
    )

    if legend:
        text += "Below is a color mapping indicating the room type for each fill color:\n"
        for rgb, room in COLOR_MAPPING.items():
            hex_code = f"#{rgb[0]:02X}{rgb[1]:02X}{rgb[2]:02X}"
            text += f"- {room}: {hex_code} (RGB{rgb})\n"
        text += "\n"
    
    text += (
        "Examine each floorplan **only within its thick black outer boundary**, and **use the main entrance (the red bar) as the point of entry to reorient yourself when reasoning about the spatial layout **, "
        "focusing on spatial layout, room types, and relative sizes.\n"
        "Question: Which floorplan (A, B, C, D, or E) has a different underlying pattern, and why?\n\n"
        "Please structure your response exactly as follows:\n"
        "1. **Different floorplan:** <A/B/C/D/E>\n"
        "2. **Why:** <brief reasoning>\n\n"
    )
    return text


# def build_2modal_prompt(options: list[str]) -> str:
#     """
#     Creates a prompt for a consolidated 2-modality input: five floorplans labeled A–E, each with a human description.
#     """
#     letters = ['A', 'B', 'C', 'D', 'E']
#     text = (
#         "I am presenting you five apartment floorplans, labeled A through E.\n"
#         "One of these plans has a different underlying floorplan pattern, while the other four share the same pattern.\n\n"
#         "Each plan has a human-annotated description as shown below.\n\n"
#         "Descriptions:\n"
#     )

#     for label, pid in zip(letters, options):
#         annotation = load_annotation(pid)
#         text += f"- {label}: {annotation}\n"

#     text += (
#         "Here I am also showing you an image with all five floorplans, labeled A through E, corresponding to the descriptions A through E.\n\n"
#         "Please examine spatial relationships, room types, and sizes based on the descriptions and images.\n\n"
#         "Which of A, B, C, D, or E has a different underlying floorplan pattern from others, and why?\n\n"
#         "Structure your reply exactly like:\n"
#         "1. **Different plan:** <A/B/C/D/E>\n"
#         "2. **Why:** <brief reasoning>\n\n"
#     )
#     return text

In [ ]:
results = []

reoriented = False

multi = True

if legend:
    f_sfx = "wlegend"
else:
    f_sfx = "nolegend"

if reoriented:
    f_suffix = "reoriented"
else:
    f_suffix = "original"

if reoriented:
    suffix = ""
else:
    suffix = "_2"

for i, row in df.iterrows():
    # Each row points to one consolidated image file, e.g. case_01.png
    case_id = i + 1  # adjust column name as needed
    options = row['options']

    img_path = data_dir / f"example_{case_id}{suffix}.png"

    # Open the single consolidated image
    consolidated_img = Image.open(img_path).convert("RGB")

    # Build prompt
    label_map = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5}  # if needed

    if multi:
        prompt = build_2modal_prompt(options)
    else:
        prompt = build_prompt(label_map)

    # Run the VLM model on the single image
    answer = vlm([consolidated_img], prompt, do_sample=False)

    # Parse out the letter from the first line
    first_line = answer.splitlines()[0]
    predicted_label = first_line.split(":", 1)[1].strip()

    results.append({
        'base_cluster':  row['base_cluster'],
        'other_cluster': row['other_cluster'],
        'options':       options,
        'outlier_id':    row['outlier_id'],
        'predicted_id':  predicted_label,
        'raw_response':  answer
    })
# save
df_out = pd.DataFrame(results)

if not multi:
    out_path = Path("outputs/vlm_qwen2.5-vl_img_{}_{}_{}.csv".format(task, f_suffix, f_sfx))
else:
    out_path = Path("outputs/vlm_qwen2.5-vl_multi_{}_{}_{}.csv".format(task, f_suffix, f_sfx))
out_path.parent.mkdir(exist_ok=True)
df_out.to_csv(out_path, index=False)
print(f"Wrote {len(df_out)} results to {out_path}")

PosixPath('../../data/floorplan_image')

In [ ]:
tasks = ['same_second', 'diff_first']
legend_opts = [True, False]
# legend_opts = [True]
orient_opts = [True, False]
multi_opts = [True, False]

# map label->int if needed
label_map = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5}

for task in tasks:
    for legend in legend_opts:
        for reoriented in orient_opts:
            for multi in multi_opts:
                # Determine suffixes and paths
                sfx = '_wlegend_bound' if legend else ''
                f_sfx = 'wlegend' if legend else 'nolegend'
                f_suffix = 'reoriented' if reoriented else 'original'
                suffix = '' if reoriented else '_2'

                csv_path = Path(f"groups_{task}_results.csv")
                data_dir = Path(f"./randomized_options/examples_{task}{sfx}")

                # Read input data
                df = pd.read_csv(csv_path)
                results = []

                for i, row in df.iterrows():
                    case_id = i + 1
                    options = row['options']
                    img_path = data_dir / f"example_{case_id}{suffix}.png"

                    # Load image
                    consolidated_img = Image.open(img_path).convert("RGB")

                    # Build prompt
                    if multi:
                        prompt = build_2modal_prompt(options, legend)
                    else:
                        prompt = build_prompt(options, legend)

                    # Query VLM
                    answer = vlm([consolidated_img], prompt, do_sample=False)

                    # Parse prediction
                    first_line = answer.splitlines()[0]
                    predicted_label = first_line.split(":", 1)[1].strip()

                    results.append({
                        'base_cluster':  row['base_cluster'],
                        'other_cluster': row['other_cluster'],
                        'options':       options,
                        'outlier_id':    row['outlier_id'],
                        'predicted_id':  predicted_label,
                        'raw_response':  answer
                    })

                # Save outputs
                df_out = pd.DataFrame(results)
                mode = 'vlm_qwen2.5-vl_w_door_orientation_bound_legendbound'
                type_str = 'multi' if multi else 'img'
                out_dir = Path('outputs') / mode
                out_dir.mkdir(parents=True, exist_ok=True)
                out_name = f"{mode}_{type_str}_{task}_{f_suffix}_{f_sfx}.csv"
                out_path = out_dir / out_name
                df_out.to_csv(out_path, index=False)
                print(f"Wrote {len(df_out)} results to {out_path}")


/home/hice1/hzhang931/scratch/planscape/venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1e-06` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


Wrote 100 results to outputs/vlm_qwen2.5-vl_w_door_orientation_bound_legendbound/vlm_qwen2.5-vl_w_door_orientation_bound_legendbound_multi_same_second_reoriented_wlegend.csv


/home/hice1/hzhang931/scratch/planscape/venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1e-06` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


Wrote 100 results to outputs/vlm_qwen2.5-vl_w_door_orientation_bound_legendbound/vlm_qwen2.5-vl_w_door_orientation_bound_legendbound_img_same_second_reoriented_wlegend.csv


/home/hice1/hzhang931/scratch/planscape/venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1e-06` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


...

/home/hice1/hzhang931/scratch/planscape/venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1e-06` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


Wrote 100 results to outputs/vlm_qwen2.5-vl_w_door_orientation_bound_legendbound/vlm_qwen2.5-vl_w_door_orientation_bound_legendbound_img_same_second_original_wlegend.csv


/home/hice1/hzhang931/scratch/planscape/venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1e-06` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


Wrote 100 results to outputs/vlm_qwen2.5-vl_w_door_orientation_bound_legendbound/vlm_qwen2.5-vl_w_door_orientation_bound_legendbound_multi_same_second_reoriented_nolegend.csv


/home/hice1/hzhang931/scratch/planscape/venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1e-06` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


In [ ]:
tasks = ['difficult', 'easy']
legend_opts = [True, False]
orientations = ['mix', 'reoriented', 'original']

# legend_opts = [True]
# orient_opts = [True, False]
# multi_opts = [True, False]

# map label->int if needed
label_map = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5}

for task in tasks:
    for legend in legend_opts:
        for orientation in orientations:

        # Determine suffixes and paths
        sfx = '_wlegend_bound' if legend else ''
        f_sfx = 'wlegend' if legend else 'nolegend'

        csv_path = Path(f"{task}_dataset_mid_train_test.csv")
        data_dir = Path(f"./datasets/{task}")

        # Read input data
        df = pd.read_csv(csv_path)
        df = df[df['split'] == 'test']
        results = []

        for i, row in df.iterrows():
            case_id = row['orig_index'] + 1
            options = row['options']
            img_path = data_dir / f"example_{case_id}_2.png"

            # Load image
            consolidated_img = Image.open(img_path).convert("RGB")

            prompt = build_prompt(options, legend)

            # Query VLM
            answer = vlm([consolidated_img], prompt, do_sample=False)

            # Parse prediction
            first_line = answer.splitlines()[0]
            predicted_label = first_line.split(":", 1)[1].strip()

            results.append({
                'base_cluster':  row['base_cluster'],
                'other_cluster': row['other_cluster'],
                'options':       options,
                'outlier_id':    row['outlier_id'],
                'predicted_id':  predicted_label,
                'raw_response':  answer
            })

        # Save outputs
        df_out = pd.DataFrame(results)
        mode = 'vlm_qwen2.5-vl_zeroshot'
        type_str = 'img'
        out_dir = Path('qwen2p5_zeroshotout') 
        out_dir.mkdir(parents=True, exist_ok=True)
        out_name = f"{mode}_{type_str}_{task}_nolegend.csv"
        out_path = out_dir / out_name
        df_out.to_csv(out_path, index=False)
        print(f"Wrote {len(df_out)} results to {out_path}")


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


IndexError: list index out of range

In [8]:
tasks = ['difficult', 'easy']
legend_opts = [True, False]
orientations = ['reoriented', 'original']
# orientations = ['mix', 'reoriented', 'original']

# legend_opts = [True]
# orient_opts = [True, False]
# multi_opts = [True, False]

# map label->int if needed
label_map = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5}

for task in tasks:
    for legend in legend_opts:
        for orientation in orientations:

            # Determine suffixes and paths
            # sfx = '_wlegend_bound' if legend else ''
            f_sfx = 'wlegend' if legend else 'nolegend'

            csv_path = Path(f"{task}_dataset_mid_train_test.csv")
            data_dir = Path(f"./datasets/{task}_{orientation}")

            # Read input data
            df = pd.read_csv(csv_path)
            df = df[df['split'] == 'test']
            results = []

            for i, row in df.iterrows():
                case_id = row['orig_index'] + 1
                options = row['options']
                img_path = data_dir / f"example_{case_id}_2.png"

                # Load image
                consolidated_img = Image.open(img_path).convert("RGB")

                prompt = build_prompt(options, legend)

                # Query VLM
                answer = vlm([consolidated_img], prompt, do_sample=False)

                # Parse prediction
                first_line = answer.splitlines()[0]
                predicted_label = first_line.strip()

                results.append({
                    'base_cluster':  row['base_cluster'],
                    'other_cluster': row['other_cluster'],
                    'options':       options,
                    'outlier_id':    row['outlier_id'],
                    'predicted_id':  predicted_label,
                    'raw_response':  answer
                })

            # Save outputs
            df_out = pd.DataFrame(results)
            mode = 'vlm_qwen2.5-vl'
            type_str = 'zeroshot'
            out_dir = Path('qwen2p5_zeroshotout') 
            out_dir.mkdir(parents=True, exist_ok=True)
            out_name = f"{mode}_{type_str}_{orientation}_{task}_{f_sfx}.csv"
            out_path = out_dir / out_name
            df_out.to_csv(out_path, index=False)
            print(f"Wrote {len(df_out)} results to {out_path}")


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

Wrote 1055 results to qwen2p5_zeroshotout/vlm_qwen2.5-vl_zeroshot_reoriented_difficult_wlegend.csv


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

Wrote 1055 results to qwen2p5_zeroshotout/vlm_qwen2.5-vl_zeroshot_original_difficult_wlegend.csv


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

...

Wrote 1030 results to qwen2p5_zeroshotout/vlm_qwen2.5-vl_zeroshot_original_easy_wlegend.csv


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

Wrote 1030 results to qwen2p5_zeroshotout/vlm_qwen2.5-vl_zeroshot_reoriented_easy_nolegend.csv


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignore

Wrote 1030 results to qwen2p5_zeroshotout/vlm_qwen2.5-vl_zeroshot_original_easy_nolegend.csv
